# C3 companion — A real CNN on chest X-rays (MedMNIST)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/5x5x5x5/taihls/blob/course-curriculum/notebooks/c3-medical-images-cnn.ipynb)

In the chapter [](../content/3_c3-medical-images.md) we treated tiny 8x8 digits as grids of numbers and classified them with a simple model. Here we train a small **convolutional neural network (CNN)** on **PneumoniaMNIST** — real pediatric chest X-rays labelled *normal* vs *pneumonia* — and watch it learn to read scans. Use a GPU on Colab (Runtime → Change runtime type → GPU).

In [ ]:
# MedMNIST ships the datasets; torch provides the network and training tools.
!pip install -q medmnist torch torchvision matplotlib

In [ ]:
# GPU check: True means a GPU is available.
import torch
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)

## 1. Load real chest X-rays

PneumoniaMNIST is part of the MedMNIST collection: thousands of 28x28 grayscale chest X-rays,
each labelled 0 (normal) or 1 (pneumonia). Like the chapter's digits, every image is just a
grid of brightness numbers — only bigger, and from a real clinic.

In [ ]:
import medmnist
from medmnist import INFO
import torchvision.transforms as T
from torch.utils.data import DataLoader

info = INFO["pneumoniamnist"]
DataClass = getattr(medmnist, info["python_class"])
print("Task:", info["task"], "| classes:", info["label"])

transform = T.Compose([T.ToTensor()])   # turn each image into a 1x28x28 tensor in 0..1
train_ds = DataClass(split="train", transform=transform, download=True)
test_ds  = DataClass(split="test",  transform=transform, download=True)
print("train images:", len(train_ds), " test images:", len(test_ds))

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

## 2. Look at a few scans

Always eyeball your data first. Each is a 28x28 grid of numbers, drawn here as an image.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 6, figsize=(11, 2))
for ax, (img, label) in zip(axes, train_ds):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title("pneumonia" if int(label) == 1 else "normal", fontsize=8)
    ax.axis("off")
plt.show()

## 3. A small convolutional network

A CNN slides small windows (**convolutions**) across the image to detect local patterns —
edges, then textures, then shapes — instead of giving every pixel its own independent weight.
That is why it reads scans far better than the flattened classifier from the chapter. This is
a deliberately small CNN so it trains quickly.

In [ ]:
import torch.nn as nn

model = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 28x28 -> 14x14
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 14x14 -> 7x7
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 64), nn.ReLU(),
    nn.Linear(64, 2),          # two outputs: normal vs pneumonia
).to(device)
print(model)

## 4. Train it

Same gradient-descent loop as the C2 notebook, just on images and across several epochs. Each
**epoch** is one full pass over the training set; the loss should fall as the CNN learns which
patterns mean pneumonia.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(3):                       # a few epochs is enough to see learning
    model.train()
    running = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.squeeze().long().to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()                      # downhill direction for every weight
        optimizer.step()                     # one step downhill
        running += loss.item()
    print(f"epoch {epoch}  avg loss {running / len(train_loader):.3f}")

## 5. Measure accuracy on held-out scans

The honest test is performance on X-rays the model never trained on — exactly the lesson from
the chapter.

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.squeeze().long().to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
print(f"Test accuracy: {correct / total:.1%}")

## Your turn

1. **Train longer.** Raise the epochs from 3 to 10. Does test accuracy keep rising, or does it
   plateau (a sign of [overfitting](../content/4_c4-how-models-learn.md))?
2. **Look at mistakes.** Find a few test images the model got wrong and display them. Can *you*
   tell which are pneumonia? This is why a human radiologist stays in the loop.
3. **Beyond accuracy.** Pneumonia is the rarer, more dangerous class to miss. Compute how many
   true pneumonia cases the model *missed* (false negatives). Why does that number matter more
   than overall accuracy in a hospital? (See [](../content/3_b3-sensitivity-specificity.md).)